# Human Action Recognition - Multimodel Training
This notebook loads the `Bingsu/Human_Action_Recognition` dataset and trains three models:
1. **ResNet** (Implemented from scratch)
2. **Google Vision Transformer (ViT)** (Pre-trained fine-tuning)
3. **EfficientNet** (Pre-trained fine-tuning)

Optimized for Kaggle **T4x2** GPUs using `nn.DataParallel`.

In [ ]:
!pip install datasets transformers timm --upgrade -q

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
import timm
from datasets import load_dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Kaggle T4x2 GPU setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()
print(f'Using {num_gpus} GPUs:', [torch.cuda.get_device_name(i) for i in range(num_gpus)])

In [ ]:
# Load the dataset
print('Loading dataset...')
dataset = load_dataset('Bingsu/Human_Action_Recognition')

# The dataset has train and test splits
# We will create a train and validation split from train
train_val = dataset['train'].train_test_split(test_size=0.1, seed=42)
train_dataset = train_val['train']
val_dataset = train_val['test']
test_dataset = dataset['test'] # Test split usually doesn't have correct labels (all 0), but we keep it

class_names = dataset['train'].features['labels'].names
num_classes = len(class_names)
print(f'Number of classes: {num_classes}')
print(f'Classes: {class_names}')

In [ ]:
# Define common transform shapes
img_size = 224

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(img_size),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def train_transforms_fn(examples):
    examples['pixel_values'] = [train_transforms(image.convert('RGB')) for image in examples['image']]
    return examples

def val_transforms_fn(examples):
    examples['pixel_values'] = [val_transforms(image.convert('RGB')) for image in examples['image']]
    return examples

train_dataset.set_transform(train_transforms_fn)
val_dataset.set_transform(val_transforms_fn)

# Scale batch size for multi-GPU setup
batch_size = 64 * max(1, num_gpus)

def collate_fn(examples):
    pixel_values = torch.stack([example['pixel_values'] for example in examples])
    labels = torch.tensor([example['labels'] for example in examples])
    return {'pixel_values': pixel_values, 'labels': labels}

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=5):
    # Wrap model for DataParallel (T4x2 support)
    if num_gpus > 1:
        model = nn.DataParallel(model)
        
    model = model.to(device)
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        loop = tqdm(train_loader, leave=False, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
        for batch in loop:
            inputs, labels = batch['pixel_values'].to(device), batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            loop.set_postfix(loss=loss.item(), acc=100.*correct/total)
            
        epoch_loss = running_loss / total
        epoch_acc = 100. * correct / total
        
        # Validation Phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            val_loop = tqdm(val_loader, leave=False, desc=f'Epoch {epoch+1}/{num_epochs} [Val]')
            for batch in val_loop:
                inputs, labels = batch['pixel_values'].to(device), batch['labels'].to(device)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                
        val_epoch_loss = val_loss / val_total
        val_epoch_acc = 100. * val_correct / val_total
        
        print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.2f}% | Val Loss: {val_epoch_loss:.4f} Acc: {val_epoch_acc:.2f}%')
        
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)
        history['val_loss'].append(val_epoch_loss)
        history['val_acc'].append(val_epoch_acc)
        
    return model, history

### 1. ResNet Implemented From Scratch
Implementing a smaller ResNet (ResNet-18 variant) to make training from scratch feasible within reasonable time.

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=15):
        super(ResNet, self).__init__()
        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_channels, out_channels, s))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.maxpool(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out

def ResNet18Scratch(num_classes):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)

resnet_model = ResNet18Scratch(num_classes)
print(f'ResNet (Scratch) parameters: {sum(p.numel() for p in resnet_model.parameters() if p.requires_grad):,}')

In [ ]:
criterion = nn.CrossEntropyLoss()
# Use a standard optimizer for scratch training
optimizer_resnet = optim.AdamW(resnet_model.parameters(), lr=1e-3, weight_decay=1e-4)

print('Training ResNet (From Scratch)...')
# Train for 5 epochs for demonstration; increase to 15-30 for actual convergence
resnet_model, history_resnet = train_model(resnet_model, train_loader, val_loader, criterion, optimizer_resnet, num_epochs=5)

### 2. Google Vision Transformer (ViT)
Using pre-trained ViT base patch 16 (224) and fine-tuning for our 15 action classes.

In [ ]:
vit_model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes)
print(f'ViT parameters: {sum(p.numel() for p in vit_model.parameters() if p.requires_grad):,}')

# Lower learning rate for pre-trained ViT fine-tuning
optimizer_vit = optim.AdamW(vit_model.parameters(), lr=3e-5, weight_decay=1e-4)

print('Training Vision Transformer (ViT)...')
vit_model, history_vit = train_model(vit_model, train_loader, val_loader, criterion, optimizer_vit, num_epochs=3)

### 3. EfficientNet
Using pre-trained EfficientNet B0 and fine-tuning.

In [ ]:
efficientnet_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=num_classes)
print(f'EfficientNet parameters: {sum(p.numel() for p in efficientnet_model.parameters() if p.requires_grad):,}')

optimizer_eff = optim.AdamW(efficientnet_model.parameters(), lr=3e-4, weight_decay=1e-4)

print('Training EfficientNet...')
efficientnet_model, history_eff = train_model(efficientnet_model, train_loader, val_loader, criterion, optimizer_eff, num_epochs=3)

### 4. Performance Comparison
Let's visualize the training and validation Accuracy and Loss over epochs across all three models.

In [ ]:
import matplotlib.pyplot as plt

def plot_history(histories, title, metric='acc'):
    plt.figure(figsize=(10, 6))
    for name, history in histories.items():
        plt.plot(history[f'val_{metric}'], label=f'{name} Val {metric.capitalize()}')
        # Optional: uncomment the line below to also show training curves
        # plt.plot(history[f'train_{metric}'], label=f'{name} Train {metric.capitalize()}', linestyle='--', alpha=0.5)
        
    plt.title(title)
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)' if metric == 'acc' else 'Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

all_histories = {
    'ResNet (Scratch)': history_resnet,
    'ViT': history_vit,
    'EfficientNet': history_eff
}

plot_history(all_histories, 'Model Accuracy Comparison', 'acc')
plot_history(all_histories, 'Model Loss Comparison', 'loss')

### 5. Exporting Models
Exporting the models into a standard format (`.pth`) so they can easily be deployed or imported later. We extract the original model from `DataParallel` to avoid compatibility issues during inference on inference systems without identical GPU topologies.

In [ ]:
import os

os.makedirs('saved_models', exist_ok=True)

def save_model(model, path):
    # Extract the original model from DataParallel if it was wrapped for T4x2
    model_to_save = model.module if isinstance(model, nn.DataParallel) else model
    # Optionally save the entire model `torch.save(model_to_save, path)`
    # Or standard best practice: save state_dict
    torch.save(model_to_save.state_dict(), path)
    print(f"Model saved to {path}")

# Save ResNet
save_model(resnet_model, 'saved_models/resnet18_scratch.pth')

# Save ViT
save_model(vit_model, 'saved_models/vit_base_patch16.pth')

# Save EfficientNet
save_model(efficientnet_model, 'saved_models/efficientnet_b0.pth')

print("\nAll models exported successfully in standard PyTorch (.pth) format!")
print("You can later load them via `model.load_state_dict(torch.load('path.pth'))`")